In [1]:
import pandas as pd # type: ignore
import numpy as np
import matplotlib as plt  # type: ignore
import seaborn as sns # type: ignore
from sklearn.model_selection import train_test_split, GridSearchCV # type: ignore
from sklearn.metrics import mean_squared_error, r2_score # type: ignore
import os
import joblib # type: ignore
from sklearn.linear_model import Lasso

In [2]:
dr = pd.read_excel('descriptors.xlsx')
del dr['ID']
dr.head()


,smiles,vmin_vmin_boltz,vmin_r_boltz,fmo_e_homo_boltz,fmo_e_lumo_boltz,fmo_mu_boltz,fmo_eta_boltz,fmo_omega_boltz,somo_ra_boltz,somo_rc_boltz,...,sterimol_burB5_boltz,sterimol_burB5_min,sterimol_burB5_max,sterimol_burB5_delta,sterimol_burB5_vburminconf,sterimol_burL_boltz,sterimol_burL_min,sterimol_burL_max,sterimol_burL_delta,sterimol_burL_vburminconf
0,CC(C)c1cc(C(C)C)c(-c2ccccc2P(C2CCCCC2)C2CCCCC2...,-0.061654,1.819048,-0.218243,-0.025112,-0.121678,0.193131,0.038336,0.059115,-0.363712,...,7.439480,6.306577,7.835474,1.528897,7.262693,7.291573,7.106519,8.238802,1.132283,7.735548
1,CN(C)c1cccc(N(C)C)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1,-0.063670,1.784157,-0.206310,-0.023277,-0.114794,0.183033,0.036033,0.061658,-0.345264,...,6.572514,6.339063,7.850955,1.511892,6.407769,7.285463,6.908743,8.216943,1.308200,7.992698
2,COc1cccc(OC)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1,-0.066303,1.798595,-0.213323,-0.016923,-0.115123,0.196400,0.033750,0.069846,-0.356952,...,7.156276,6.349086,7.287324,0.938238,7.021625,7.306643,7.025374,8.361404,1.336030,7.424874
3,CC(C)Oc1cccc(OC(C)C)c1-c1ccccc1P(C1CCCCC1)C1CC...,-0.067319,1.795292,-0.211571,-0.013802,-0.112687,0.197770,0.032124,0.070775,-0.349017,...,7.238774,6.369287,7.813010,1.443723,7.653520,7.338674,6.996319,8.333428,1.337110,7.483233
4,c1ccc(-c2ccccc2P(C2CCCCC2)C2CCCCC2)cc1,-0.061351,1.816461,-0.218842,-0.030145,-0.124494,0.188697,0.041069,0.060353,-0.376125,...,6.497622,6.092458,7.055260,0.962802,6.376705,7.370445,7.021018,8.155038,1.134020,8.086289


In [3]:
df = pd.read_csv("Morgan.csv")
del df['ID']
df.head()

,smiles,morgan_fingerprint
0,CC(C)c1cc(C(C)C)c(-c2ccccc2P(C2CCCCC2)C2CCCCC2...,0110100000000000000000000000000101000000000001...
1,CN(C)c1cccc(N(C)C)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1,0010100000000001000000000000000101000000000001...
2,COc1cccc(OC)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1,0010100000000001000000000000001101000000000001...
3,CC(C)Oc1cccc(OC(C)C)c1-c1ccccc1P(C1CCCCC1)C1CC...,0110100000000001000000000000001101000000000001...
4,c1ccc(-c2ccccc2P(C2CCCCC2)C2CCCCC2)cc1,0010100000000000000000000000000100000000000001...


In [4]:
def split_bits(row):
    if pd.isna(row['morgan_fingerprint']):
        return pd.Series([np.nan] * 1024)
    else:
        return pd.Series(list(row['morgan_fingerprint']))

bit_columns = df.apply(split_bits, axis=1)

bit_columns.columns = [f'F_{i+1}' for i in range(bit_columns.shape[1])]
final_df = pd.concat([df.reset_index(drop=True), bit_columns], axis=1)
final_df.head()

,smiles,morgan_fingerprint,F_1,F_2,F_3,F_4,F_5,F_6,F_7,F_8,...,F_1015,F_1016,F_1017,F_1018,F_1019,F_1020,F_1021,F_1022,F_1023,F_1024
0,CC(C)c1cc(C(C)C)c(-c2ccccc2P(C2CCCCC2)C2CCCCC2...,0110100000000000000000000000000101000000000001...,0,1,1,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,CN(C)c1cccc(N(C)C)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1,0010100000000001000000000000000101000000000001...,0,0,1,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,COc1cccc(OC)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1,0010100000000001000000000000001101000000000001...,0,0,1,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,CC(C)Oc1cccc(OC(C)C)c1-c1ccccc1P(C1CCCCC1)C1CC...,0110100000000001000000000000001101000000000001...,0,1,1,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,c1ccc(-c2ccccc2P(C2CCCCC2)C2CCCCC2)cc1,0010100000000000000000000000000100000000000001...,0,0,1,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [5]:
final_df = final_df.dropna()

y1 = dr[dr['smiles'].isin(final_df['smiles'])]

In [6]:
input = final_df.copy()
del input['morgan_fingerprint']
del input['smiles']
input.head()

,F_1,F_2,F_3,F_4,F_5,F_6,F_7,F_8,F_9,F_10,...,F_1015,F_1016,F_1017,F_1018,F_1019,F_1020,F_1021,F_1022,F_1023,F_1024
0,0,1,1,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,0,0,1,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,0,0,1,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0,1,1,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,0,0,1,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [7]:
del y1['smiles']

In [8]:
for column in input.columns:
    if input[column].dtype == 'object':
        input[column] = pd.to_numeric(input[column], errors='coerce')

# Check the new data types
print(input.dtypes)

F_1       int64
F_2       int64
F_3       int64
F_4       int64
F_5       int64
          ...  
F_1020    int64
F_1021    int64
F_1022    int64
F_1023    int64
F_1024    int64
Length: 1024, dtype: object


In [9]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


In [10]:
n_components = 5

abb = y1

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(abb)

#pca = PCA(n_components=n_components)
#X_pca = pca.fit_transform(X_scaled)

# Create a DataFrame for the PCA-transformed data
#pca_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(n_components)])

# Save the PCA-transformed data to a new CSV file
#pca_df.to_csv('pca.csv', index=False)

print("Transformation complete. Transformed data saved to 'pca.csv'.")


Transformation complete. Transformed data saved to 'pca.csv'.


In [11]:
pip install optuna

Note: you may need to restart the kernel to use updated packages.


In [12]:
import logging
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import optuna
import joblib

In [13]:
from tensorflow.keras.layers import BatchNormalization

In [27]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

X = input
Y = abb.copy()  
results_list = []
model_dir = 'models_dl_bayes_val_cv'
os.makedirs(model_dir, exist_ok=True)

# Function to create the model
def create_model(n_layers, n_units, dropout_rate, learning_rate, input_shape):
    model = Sequential()
    model.add(Dense(n_units, activation='relu', input_shape=(input_shape,)))
    model.add(BatchNormalization())
    
    for _ in range(n_layers - 1):
        model.add(Dense(n_units, activation='relu'))
        model.add(BatchNormalization())
        model.add(Dropout(dropout_rate))
    
    model.add(Dense(1))  # Output layer for regression

    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mean_absolute_error')
    return model

# Optuna objective function
def objective(trial, X_train, y_train, X_val, y_val):
    # Hyperparameters to tune
    n_layers = trial.suggest_int('n_layers', 1, 5)
    n_units = trial.suggest_int('n_units', 16, 256)
    dropout_rate = trial.suggest_float('dropout_rate', 0.0, 0.5)
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-2)
    batch_size = trial.suggest_int('batch_size', 16, 128)

    # Create the model
    model = create_model(n_layers, n_units, dropout_rate, learning_rate, X_train.shape[1])

    # Train the model
    history = model.fit(X_train, y_train, 
                        validation_data=(X_val, y_val), 
                        epochs=100, batch_size=batch_size, 
                        verbose=0, callbacks=[tf.keras.callbacks.EarlyStopping(patience=5)])
    
    # Evaluate the model
    val_pred = model.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_pred)

    return val_mae  # Minimize MAE

for feature in Y.columns:
    model_path = os.path.join(model_dir, f'model_{feature}.h5')

    # Skip if model already exists
    if os.path.exists(model_path):
        logging.info(f"Model for feature '{feature}' already exists, skipping...")
        continue

    y = Y[feature].to_numpy()  # Convert y to a NumPy array
    
    # Split the data into train, validation, and test sets
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

    # Ensure types are correct
    X_train = X_train.astype(np.float64)
    y_train = y_train.astype(np.float64)

    # Log shapes
    logging.info("Shapes for feature '%s': X_train: %s, y_train: %s", feature, X_train.shape, y_train.shape)

    # Create a study for Bayesian optimization
    study = optuna.create_study(direction='minimize')  # Minimize MAE
    try:
        study.optimize(lambda trial: objective(trial, X_train, y_train, X_val, y_val), n_trials=10)
    except Exception as e:
        logging.error(f"Error during optimization for feature '{feature}': {e}")
        continue
    
    best_params = study.best_params
    
    # Create the best model with optimized parameters
    best_model = create_model(
        best_params['n_layers'], 
        best_params['n_units'], 
        best_params['dropout_rate'], 
        best_params['learning_rate'], 
        X_train.shape[1]
    )
    
    # Fit the best model
    try:
        best_model.fit(X_train, y_train, 
                       validation_data=(X_val, y_val), 
                       epochs=100, batch_size=best_params['batch_size'], 
                       verbose=0, callbacks=[tf.keras.callbacks.EarlyStopping(patience=5)])
    except Exception as e:
        logging.error(f"Error during fitting of the best model for feature '{feature}': {e}")
        continue

    # Evaluate the model on validation and test sets
    val_pred = best_model.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_pred)  # Validation MAE
    
    y_pred = best_model.predict(X_test)
    test_mae = mean_absolute_error(y_test, y_pred)  # Test MAE
    r2 = r2_score(y_test, y_pred)
    
    # Store the results
    results_list.append({
        'Feature': feature,
        'Validation MAE': val_mae,
        'Test MAE': test_mae,
        'R²': r2,
        'Best Parameters': best_params
    })
    
    # Save the model
    best_model.save(model_path)
    logging.info(f"Model for feature '{feature}' saved successfully.")

# Save all results to a CSV file
results_df = pd.DataFrame(results_list)
results_df.to_csv('model_performance_dl.csv', index=False)
logging.info("Best model performance metrics saved to 'model_performance_dl.csv'.")
logging.info("Best models saved in the 'models_dl_bayes_val_cv' directory.")


2024-09-26 19:06:11,074 - INFO - Model for feature 'vmin_vmin_boltz' already exists, skipping...
2024-09-26 19:06:11,077 - INFO - Model for feature 'vmin_r_boltz' already exists, skipping...
2024-09-26 19:06:11,080 - INFO - Model for feature 'fmo_e_homo_boltz' already exists, skipping...
2024-09-26 19:06:11,081 - INFO - Model for feature 'fmo_e_lumo_boltz' already exists, skipping...
2024-09-26 19:06:11,081 - INFO - Model for feature 'fmo_mu_boltz' already exists, skipping...
2024-09-26 19:06:11,081 - INFO - Model for feature 'fmo_eta_boltz' already exists, skipping...
2024-09-26 19:06:11,086 - INFO - Model for feature 'fmo_omega_boltz' already exists, skipping...
2024-09-26 19:06:11,087 - INFO - Model for feature 'somo_ra_boltz' already exists, skipping...
2024-09-26 19:06:11,090 - INFO - Model for feature 'somo_rc_boltz' already exists, skipping...
2024-09-26 19:06:11,091 - INFO - Model for feature 'nbo_P_boltz' already exists, skipping...
2024-09-26 19:06:11,093 - INFO - Model for f

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step


[I 2024-09-26 19:06:29,303] Trial 0 finished with value: 0.39816161618356966 and parameters: {'n_layers': 4, 'n_units': 244, 'dropout_rate': 0.11418895952590641, 'learning_rate': 0.0003673437183215118, 'batch_size': 89}. Best is trial 0 with value: 0.39816161618356966.
C:\Users\UTKARSH GUPTA\AppData\Local\Temp\ipykernel_18756\4112935901.py:31: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-2)
D:\conda\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


[I 2024-09-26 19:06:35,654] Trial 1 finished with value: 0.4144556965334086 and parameters: {'n_layers': 1, 'n_units': 24, 'dropout_rate': 0.11884645695021734, 'learning_rate': 0.0005599465537515061, 'batch_size': 75}. Best is trial 0 with value: 0.39816161618356966.
C:\Users\UTKARSH GUPTA\AppData\Local\Temp\ipykernel_18756\4112935901.py:31: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-2)
D:\conda\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

[I 2024-09-26 19:10:05,589] Trial 8 finished with value: 0.32666958469861557 and parameters: {'n_layers': 5, 'n_units': 91, 'dropout_rate': 0.12307118890353214, 'learning_rate': 0.0032961546827881957, 'batch_size': 73}. Best is trial 1 with value: 0.30775151700990827.
C:\Users\UTKARSH GUPTA\AppData\Local\Temp\ipykernel_18756\4112935901.py:31: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-2)
D:\conda\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step


[I 2024-09-26 19:10:11,573] Trial 9 finished with value: 6.362979532033261 and parameters: {'n_layers': 2, 'n_units': 44, 'dropout_rate': 0.04253330800344379, 'learning_rate': 0.0001170117784801829, 'batch_size': 30}. Best is trial 1 with value: 0.30775151700990827.
D:\conda\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


2024-09-26 19:10:19,533 - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
2024-09-26 19:10:19,577 - INFO - Model for feature 'sterimol_burL_vburminconf' saved successfully.
2024-09-26 19:10:19,588 - INFO - Best model performance metrics saved to 'model_performance_dl.csv'.
2024-09-26 19:10:19,588 - INFO - Best models saved in the 'models_dl_bayes_val_cv' directory.


In [31]:
from keras.models import load_model


In [57]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

X = input
Y = abb.copy() 

# Split the data into train, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, Y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Define the model directory and result storage
model_dir = 'models_dl_bayes_val_cv'
results_list = []

 

# Iterate through each model file in the model directory
for model_file in os.listdir(model_dir):
    if model_file.endswith('.h5'):
        print(model_file)
        # Extract the feature name from the filename
        feature = model_file[6:-3]  # Remove 'model_' from the start and '.h5' from the end
        model_path = os.path.join(model_dir, model_file)
        # Load the model
        model = load_model(model_path)

        # Predict using the model on the test set
        y_pred = model.predict(X_test)
        
        # Log shapes and types
        logging.info("Shapes: y_test: %s, y_pred: %s", y_test.shape, y_pred.shape)
        logging.info("Types: y_test: %s, y_pred: %s", type(y_test), type(y_pred))
        
        # Calculate MAE and R²
        test_mae = mean_absolute_error(y_test[feature], y_pred)
        r2 = r2_score(y_test[feature], y_pred)

        
        # Store the results
        results_list.append({
            'Feature': feature,
            'Test MAE': test_mae,
            'R²': r2
        })
        
        logging.info(f"Model for feature '{feature}' evaluated. Test MAE: {test_mae}, R²: {r2}")

# Save all results to a CSV file
results_df = pd.DataFrame(results_list)
results_df.to_csv('model_evaluation_metrics.csv', index=False)
logging.info("Model evaluation metrics saved to 'model_evaluation_metrics.csv'.")


2024-09-26 19:22:55,201 - WARNING - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


model_dipolemoment_boltz.h5
dipolemoment_boltz
dipolemoment_boltz
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step


2024-09-26 19:22:55,570 - INFO - Shapes: y_test: (308, 190), y_pred: (308, 1)
2024-09-26 19:22:55,573 - INFO - Types: y_test: <class 'pandas.core.frame.DataFrame'>, y_pred: <class 'numpy.ndarray'>
2024-09-26 19:22:55,574 - INFO - Model for feature 'dipolemoment_boltz' evaluated. Test MAE: 0.5146547691934446, R²: 0.5961482435182335


model_dipolemoment_delta.h5
dipolemoment_delta
dipolemoment_delta


2024-09-26 19:22:55,822 - WARNING - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


...

2024-09-26 19:24:40,048 - INFO - Shapes: y_test: (308, 190), y_pred: (308, 1)
2024-09-26 19:24:40,049 - INFO - Types: y_test: <class 'pandas.core.frame.DataFrame'>, y_pred: <class 'numpy.ndarray'>
2024-09-26 19:24:40,052 - INFO - Model for feature 'vmin_vmin_boltz' evaluated. Test MAE: 0.017812559883669032, R²: -1.0488371156915859


model_volume_boltz.h5
volume_boltz
volume_boltz


2024-09-26 19:24:40,162 - WARNING - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


2024-09-26 19:24:40,460 - INFO - Shapes: y_test: (308, 190), y_pred: (308, 1)
2024-09-26 19:24:40,460 - INFO - Types: y_test: <class 'pandas.core.frame.DataFrame'>, y_pred: <class 'numpy.ndarray'>
2024-09-26 19:24:40,460 - INFO - Model for feature 'volume_boltz' evaluated. Test MAE: 54.15265360157201, R²: 0.7481403998093185
2024-09-26 19:24:40,467 - INFO - Model evaluation metrics saved to 'model_evaluation_metrics.csv'.
